# Faruq-v3 — ACMC1 residual error attribution (3 seeds)

Validation-only audit untuk model terpilih **ACMC1**. Tidak ada training dan tidak ada akses ke test.

Audit menggabungkan:
- AP per kelas pada IoU 0.50–0.95;
- AP50, AP75, AP95, dan penurunan AP50→AP75;
- proposal accessibility dan matched recall pada IoU 0.50;
- classification accuracy setelah objek sudah terlokalisasi/matched;
- directional confusion pairs.

Tujuan: menentukan apakah residual error ACMC1 masih dominan klasifikasi/ranking, sudah bergeser ke high-IoU localization, atau campuran.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, subprocess, sys, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/acmc1-residual-error-audit'
os.chdir('/content')
if REPO.exists():
    shutil.rmtree(REPO)

clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH,
         'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0:
        break
    if REPO.exists():
        shutil.rmtree(REPO)
    if attempt == 3:
        raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('BRANCH:', BRANCH)


In [ ]:
import tarfile, torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'

PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed123/weights/best.pt',
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed2026/weights/best.pt',
))

ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
CKPT42 = require_project_artifact(
    PROJECT_ROOT,
    'experiments/faruq-v3-acmc-one-stage-v1/ACMC1_seed42/weights/best.pt'
)
CKPT123 = require_project_artifact(
    PROJECT_ROOT,
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed123/weights/best.pt'
)
CKPT2026 = require_project_artifact(
    PROJECT_ROOT,
    'experiments/faruq-v3-acmc-paired-confirmation-v1/ACMC1/ACMC1_seed2026/weights/best.pt'
)

DATA_ROOT = Path('/content/faruq-development-v3-grouped')
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-acmc1-residual-error-audit-v1'

if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')

assert (DATA_ROOT / 'data.yaml').is_file(), DATA_ROOT
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
assert all(p.is_file() for p in (CKPT42, CKPT123, CKPT2026))

print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('DATA      :', DATA_ROOT)
print('SEED42    :', CKPT42)
print('SEED123   :', CKPT123)
print('SEED2026  :', CKPT2026)
print('OUTPUT    :', OUTPUT_ROOT)


In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_acmc1_residual_error_audit.py'],
    cwd=REPO,
    check=True,
)
print('Protocol/unit tests: PASS')


In [ ]:
command = [
    sys.executable, '-u', '-m',
    'coffee_detector.analysis.acmc1_residual_error_audit',
    '--seed42-checkpoint', str(CKPT42),
    '--seed123-checkpoint', str(CKPT123),
    '--seed2026-checkpoint', str(CKPT2026),
    '--data-root', str(DATA_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--device', '0',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
subprocess.run(command, cwd=REPO, check=True)


In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'acmc1_residual_error_attribution.json'
CSV = OUTPUT_ROOT / 'acmc1_residual_error_attribution.csv'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))

assert result['training_executed'] is False
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
assert result['test_opened'] is False
assert result['selected_model'] == 'ACMC1'
assert result['seeds'] == [42, 123, 2026]

global_rows = []
for metric, stats in result['global_aggregate'].items():
    global_rows.append({'metric': metric, **stats})
global_table = pd.DataFrame(global_rows)

class_table = pd.DataFrame(result['per_class_aggregate'])
show_cols = [
    'class_name',
    'map50_95_mean',
    'ap50_mean',
    'ap75_mean',
    'ap95_mean',
    'ap50_to_ap75_drop_mean',
    'proposal_accessibility_iou50_mean',
    'matched_recall_iou50_mean',
    'class_accuracy_given_iou50_match_mean',
    'classification_headroom_iou50_mean',
    'attribution',
    'attribution_seed_agreement',
]

display(global_table.style.format({'mean':'{:.2%}', 'std':'{:.2%}'}))
display(class_table[show_cols].style.format({
    'map50_95_mean':'{:.2%}',
    'ap50_mean':'{:.2%}',
    'ap75_mean':'{:.2%}',
    'ap95_mean':'{:.2%}',
    'ap50_to_ap75_drop_mean':'{:.2%}',
    'proposal_accessibility_iou50_mean':'{:.2%}',
    'matched_recall_iou50_mean':'{:.2%}',
    'class_accuracy_given_iou50_match_mean':'{:.2%}',
    'classification_headroom_iou50_mean':'{:.2%}',
}))

print('\nATTRIBUTION COUNTS:', result['attribution_counts'])
print('FROZEN THRESHOLDS :', result['frozen_thresholds'])
print('SUMMARY           :', SUMMARY)
print('CSV               :', CSV)
print('\nKirim global table, 10 kelas terbawah, attribution counts, dan top confusion pairs.')
print('Jangan membuka test dan jangan training model baru.')


In [ ]:
for seed in ('42', '123', '2026'):
    print(f'\n=== TOP CONFUSIONS SEED {seed} ===')
    display(pd.DataFrame(result['per_seed'][seed]['top_directional_confusions']).head(15))
